# Getting Started

**A reproducible pipeline for molecular QTL analysis — from raw genotypes and phenotypes through discovery, fine-mapping, and integration with GWAS.**

This guide takes you from a clean machine to your first successful run in about an hour.

::::{grid} 1 1 3 3
:gutter: 3

:::{grid-item-card} 📦 Install
:link: #step-1-install-pixi
One installer for Python, R, JupyterLab, and bioinformatics tools via [pixi](https://pixi.sh/).
:::

:::{grid-item-card} 🧬 Run
:link: #step-5-run-your-first-workflow
Clone the repo, grab demo data, and launch a cis-QTL scan.
:::

:::{grid-item-card} 🚀 Go Further
:link: #what-to-do-next
Fine-mapping, multivariate analysis, GWAS integration, HPC templates.
:::

::::


---

## Before You Start

You'll need a Linux or macOS shell. Windows users: install [WSL2](https://learn.microsoft.com/windows/wsl/install) first.

| Requirement | Minimum | Recommended |
|---|---|---|
| Disk space | 10 GB (minimal install) | 40 GB (full bioinformatics stack) |
| Memory | 16 GB | 50 GB+ on HPC for the installer |
| Network | GitHub, conda-forge, synapse.org | Same |
| Git | Any recent version | 2.30+ |

:::{admonition} On HPC? Start on a compute node.
:class: tip
The installer is memory-hungry and login nodes will kill it. Grab an interactive session first:

```bash
srun --mem=50G --pty bash          # SLURM
bsub -Is -M 50000 -n 4 bash        # LSF
```
:::


---

## Step 1 — Install pixi

We manage every dependency — Python, R, JupyterLab, bioinformatics tools — with [pixi](https://pixi.sh/). One installer sets it all up.

```bash
curl -fsSL https://raw.githubusercontent.com/StatFunGen/pixi-setup/refs/heads/main/pixi-setup.sh -o pixi-setup.sh
bash pixi-setup.sh
```

The installer will prompt you for two choices:

:::{dropdown} 1. Installation path
:open:
Where pixi stores environments and the package cache.

| Setting | When to use |
|---|---|
| `$HOME/.pixi` (default) | Laptops and workstations with plenty of home-directory space |
| `/lab/$USER/.pixi` or scratch | HPC systems with strict home-directory quotas |

:::

:::{dropdown} 2. Installation type
:open:
Pick based on what you plan to do.

| Type | Size | Files | Includes |
|---|---|---|---|
| **1. minimal** | ~5 GB | ~100k | CLI tools, Python data-science stack, JupyterLab, base R |
| **2. full** | ~35 GB | ~350k | Everything above, **plus** samtools, bcftools, plink2, GATK4, STAR, Seurat, Bioconductor |

Choose **minimal** for xQTL runs with pre-processed inputs; choose **full** if you'll also do upstream QC, alignment, or single-cell work.
:::

**Activate and verify:**

```bash
source ~/.bashrc          # or ~/.zshrc on macOS
pixi --version
```

You should see a version number. If not, open a fresh terminal.


---

## Step 2 — Add SoS

The protocol's pipelines are written as [SoS (Script of Scripts)](https://vatlab.github.io/sos-docs/) workflows. Install the SoS suite into pixi's Python environment:

```bash
pixi global install --environment python -c conda-forge \
    sos sos-pbs sos-notebook jupyterlab-sos \
    sos-bash sos-python sos-r

pixi run -e python python -m sos_notebook.install
```

**Verify:**

```bash
sos --version
jupyter kernelspec list    # should include 'sos'
```


---

## Step 3 — Clone the Protocol

```bash
git clone https://github.com/StatFunGen/xqtl-protocol.git
cd xqtl-protocol
```

:::{admonition} What's in the repo?
:class: note

| Folder | Contents |
|---|---|
| `pipeline/` | The SoS workflows you'll run |
| `code/` | Notebook documentation (this page lives here) |
| `data/` | Small example inputs and configuration templates |
| `website/` | JupyterBook sources for the docs site |
:::


---

## Step 4 — Download the Demo Data

The demo dataset lives on [Synapse](https://www.synapse.org/#!Synapse:syn36416559). Create a free account first, then:

```bash
pixi global install -c conda-forge --environment python synapseclient
synapse login -p
synapse get -r syn36416559 --downloadLocation data/example/
```


---

## Step 5 — Run Your First Workflow

Confirm SoS can see the pipelines:

```bash
sos run pipeline/1_xqtl_association.ipynb -h
```

You should see a list of workflow options. Now run a minimal cis-QTL scan:

```bash
sos run pipeline/TensorQTL.ipynb cis \
    --genotype-file   data/example/genotype.bed \
    --phenotype-file  data/example/phenotype.bed.gz \
    --covariate-file  data/example/covariates.tsv \
    --cwd output/demo_tensorqtl
```

Results land in `output/demo_tensorqtl/`.

:::{tip}
Every pipeline supports `-h` and prints the shell commands it runs under the hood — a great way to learn what's happening and debug failures.
:::


---

## What to Do Next

::::{grid} 1 2 2 2
:gutter: 3

:::{grid-item-card} 🔬 Preprocess your data
`1_phenotype_preprocessing.ipynb`
`2_genotype_preprocessing.ipynb`
`4_covariates_preprocessing.ipynb`
:::

:::{grid-item-card} 🧭 Discover QTLs
`TensorQTL.ipynb`
`1_xqtl_association.ipynb`
`APEX.ipynb`
:::

:::{grid-item-card} 🎯 Fine-map
`SuSiE.ipynb`
`mvSuSiE.ipynb`
`fSuSiE.ipynb`
:::

:::{grid-item-card} 🔗 Integrate with GWAS
`coloc.ipynb`
`cTWAS.ipynb`
`GWAS_integration.ipynb`
:::

::::

Full documentation: [statfungen.github.io/xqtl-protocol](https://statfungen.github.io/xqtl-protocol/).


---

## Troubleshooting

:::{dropdown} `pixi: command not found` after install
Open a new terminal, or re-source your shell rc file:
```bash
source ~/.bashrc      # Linux / HPC
source ~/.zshrc       # macOS
```
:::

:::{dropdown} Installer killed on HPC
You're running on a login node. Request a compute node with at least 50 GB of memory and re-run the installer:
```bash
srun --mem=50G --pty bash
bash pixi-setup.sh
```
:::

:::{dropdown} `sos: command not found`
Step 2 didn't complete. Re-run the `pixi global install` command and make sure `jupyter kernelspec list` shows the `sos` kernel.
:::

:::{dropdown} `ModuleNotFoundError` during a pipeline
Install the missing package into pixi's python environment:
```bash
pixi global install -c conda-forge --environment python <package>
```
:::

:::{dropdown} R package conflicts or install failures
Prefer conda-forge R packages over `install.packages()`:
```bash
pixi global install --environment r-base r-<pkg>
```
Mixing CRAN builds with conda R leads to ABI mismatches — avoid it.
:::

:::{dropdown} Still stuck?
[Open an issue](https://github.com/StatFunGen/xqtl-protocol/issues) with the command you ran and the full error output.
:::


## Analysis Overview

The protocol is modular. Each numbered pipeline is a self-contained SoS notebook that can run independently or chained together.

::::{grid} 1 2 2 2
:gutter: 3

:::{grid-item-card} 1. Preprocess
`1_phenotype_preprocessing.ipynb` — QC, normalization
`2_genotype_preprocessing.ipynb` — variant QC, imputation
`4_covariates_preprocessing.ipynb` — PEER / hidden factors
:::

:::{grid-item-card} 2. Discover
`TensorQTL.ipynb` — cis/trans scans
`APEX.ipynb` — interaction QTLs
`1_xqtl_association.ipynb` — end-to-end wrapper
:::

:::{grid-item-card} 3. Fine-map
`SuSiE.ipynb` — single-context credible sets
`mvSuSiE.ipynb` — multi-context
`fSuSiE.ipynb` — functional annotations
:::

:::{grid-item-card} 4. Integrate
`coloc.ipynb` — colocalization with GWAS
`cTWAS.ipynb` — causal TWAS
`GWAS_integration.ipynb` — joint reporting
:::

::::

All pipelines share a common config layout, so once you know one you can read the rest.
